Overview & Teori Dasar
Pada bab ini, kita membangun sebuah mini deep learning framework (seperti PyTorch atau TensorFlow) dari nol yang mendukung Automatic Differentiation (Autograd):

Tensor Abstraction: Kelas pembungkus (wrapper) untuk array NumPy yang menyimpan nilai data, gradien, serta melacak riwayat operasi matematika.

Computation Graph: Grafik terarah yang merekam operasi antar-tensor untuk menentukan alur turunan/gradien.

Automatic Backpropagation: Fungsi .backward() yang secara otomatis menghitung dan mengalirkan gradien dari rantai operasi (chain rule) tanpa perlu menuliskan formula turunan manual.

1. Membuat Kelas Tensor Sederhana dengan Autograd

In [1]:
import numpy as np

class Tensor:
    def __init__(self, data, autograd=False, creators=None, creation_op=None, id=None):
        self.data = np.array(data)
        self.autograd = autograd
        self.grad = None

        if id is None:
            self.id = np.random.randint(0, 100000)
        else:
            self.id = id

        self.creators = creators
        self.creation_op = creation_op
        self.children = {}

        # Melacak jumlah anak pada grafik komputasi
        if creators is not None:
            for c in creators:
                if self.id not in c.children:
                    c.children[self.id] = 1
                else:
                    c.children[self.id] += 1

    def all_children_grads_accounted_for(self):
        for child_id, cnt in self.children.items():
            if cnt != 0:
                return False
        return True

    def backward(self, grad=None, grad_origin=None):
        if not self.autograd:
            return

        if grad is None:
            grad = Tensor(np.ones_like(self.data))

        if grad_origin is not None:
            if self.children[grad_origin.id] == 0:
                raise Exception("Tidak dapat mengalirkan gradien dari anak yang sama lebih dari sekali!")
            else:
                self.children[grad_origin.id] -= 1

        if self.grad is None:
            self.grad = grad
        else:
            self.grad.data += grad.data

        # Hanya alirkan gradien ke creator jika seluruh anak sudah menyetorkan gradiennya
        if self.creators is not None and (self.all_children_grads_accounted_for() or grad_origin is None):
            if self.creation_op == "add":
                self.creators[0].backward(self.grad, self)
                self.creators[1].backward(self.grad, self)

    def __add__(self, other):
        if self.autograd and other.autograd:
            return Tensor(self.data + other.data,
                          autograd=True,
                          creators=[self, other],
                          creation_op="add")
        return Tensor(self.data + other.data)

    def __repr__(self):
        return f"Tensor({self.data})"

Penjelasan Teoretis:

Creators & Creation Op: Menyiapkan rekam jejak (history) dari mana suatu Tensor dihasilkan (misal: operasi penjumlahan add).

Autograd Engine: Saat .backward() dipanggil, gradien akan dialirkan mundur secara otomatis ke seluruh Tensor pembuatnya (creators).

2. Pengujian Automatic Differentiation

In [2]:
# Membuat Tensor dengan fitur autograd diaktifkan
a = Tensor([1, 2, 3, 4, 5], autograd=True)
b = Tensor([2, 2, 2, 2, 2], autograd=True)
c = Tensor([5, 4, 3, 2, 1], autograd=True)

# Computation Graph: d = a + b + c
d = a + b
e = d + c

# Panggil pembalikan gradien otomatis
e.backward(Tensor([1, 1, 1, 1, 1]))

print("Nilai Tensor e:", e)
print("Gradien Tensor a (a.grad):", a.grad)
print("Gradien Tensor b (b.grad):", b.grad)

Nilai Tensor e: Tensor([8 8 8 8 8])
Gradien Tensor a (a.grad): Tensor([1 1 1 1 1])
Gradien Tensor b (b.grad): Tensor([1 1 1 1 1])


Penjelasan Teoretis:

Dengan sistem Autograd, pengembang tidak perlu lagi menurunkan rumus kalkulus secara manual saat membangun arsitektur deep learning yang sangat kompleks.